# Assignment 2

**COMP3361 Natural Language Processing**

**Student Name:** Pang Tin Hei  
**University Number:** 3036100179

In [ ]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from seqeval.metrics import f1_score as seq_f1_score
from tqdm import tqdm
import os
import math

### Import Data

In [18]:
# Configurations
DATA_DIR = "./ontonotes5/dataset"
BATCH_SIZE = 32
MAX_LEN = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load labels mapping
with open(os.path.join(DATA_DIR, "label.json"), "r") as f:
    label2id = json.load(f)
id2label = {int(v): k for k, v in label2id.items()}
num_labels = len(label2id)


def load_data(file_paths):
    data = []
    for fp in file_paths:
        with open(os.path.join(DATA_DIR, fp), "r") as f:
            for line in f:
                data.append(json.loads(line))
    return data


train_files = ["train00.json", "train01.json", "train02.json", "train03.json"]
train_data = load_data(train_files)
valid_data = load_data(["valid.json"])
test_data = load_data(["test.json"])

# Build Vocabulary
vocab = {"<PAD>": 0, "<UNK>": 1}
for ex in train_data:
    for word in ex["tokens"]:
        if word not in vocab:
            vocab[word] = len(vocab)


class NERDataset(Dataset):
    def __init__(self, data, vocab, max_len):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = [self.vocab.get(w, self.vocab["<UNK>"]) for w in item["tokens"]]
        tags = [int(t) for t in item["tags"]]

        # Pading/Truncating
        if len(tokens) > self.max_len:
            tokens = tokens[: self.max_len]
            tags = tags[: self.max_len]
        else:
            pad_len = self.max_len - len(tokens)
            tokens.extend([self.vocab["<PAD>"]] * pad_len)
            tags.extend([-100] * pad_len)

        return torch.tensor(tokens), torch.tensor(tags)


train_dataset = NERDataset(train_data, vocab, MAX_LEN)
valid_dataset = NERDataset(valid_data, vocab, MAX_LEN)
test_dataset = NERDataset(test_data, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

Using device: cuda


### BiLSTM Tagger

In [30]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, label_size, embed_dim, hidden_dim, dropout_rate=0.3):
        super(BiLSTMTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout_rate)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim // 2,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=dropout_rate,
        )
        self.fc = nn.Linear(hidden_dim, label_size)

    def forward(self, input_ids):
        embedded = self.dropout(self.embedding(input_ids))
        lstm_out, _ = self.lstm(embedded)
        emissions = self.fc(self.dropout(lstm_out))
        return emissions
    

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for input_ids, labels in tqdm(loader):
        input_ids, labels = input_ids.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits.view(-1, num_labels), labels.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def export_predictions(model_name, predictions):
    UID = 3036100179
    output_file = f"{UID}.{model_name}.test.txt"
    with open(output_file, "w") as f:
        for ex, preds in zip(test_data, predictions):
            new_ex = {
                "tags": preds,
                "tokens": ex["tokens"]
            }
            f.write(json.dumps(new_ex) + "\n")


def evaluate(model, loader, export_prediction=False, model_name="model"):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids, labels = input_ids.to(device), labels.to(device)
            logits = model(input_ids)
            preds = torch.argmax(logits, dim=-1)
            for i in range(labels.size(0)):
                true_tags = [id2label[t.item()] for t in labels[i] if t.item() != -100]
                pred_tags = [
                    id2label[p.item()]
                    for p, t in zip(preds[i], labels[i])
                    if t.item() != -100
                ]
                y_true.append(true_tags)
                y_pred.append(pred_tags)
    if export_prediction:
        export_predictions(model_name, y_pred)
    return seq_f1_score(y_true, y_pred)


# Hyperparameters
EMBED_DIM = 64
HIDDEN_DIM = 128
EPOCHS = 5
LEARNING_RATE = 1e-2
DROPOUT_RATE = 0.1

lstm_model = BiLSTMTagger(len(vocab), num_labels, EMBED_DIM, HIDDEN_DIM, dropout_rate=DROPOUT_RATE).to(device)
print(lstm_model)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)

print("Training BiLSTM Tagger...")
for epoch in range(EPOCHS):
    loss = train_epoch(lstm_model, train_loader, optimizer)
    f1 = evaluate(lstm_model, valid_loader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Valid F1 = {f1:.4f}")

test_f1 = evaluate(lstm_model, test_loader, export_prediction=True, model_name="lstm")
print(f"\nTest F1 BiLSTM = {test_f1:.4f}")

BiLSTMTagger(
  (embedding): Embedding(41326, 64, padding_idx=0)
  (dropout): Dropout(p=0.1, inplace=False)
  (lstm): LSTM(64, 64, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
  (fc): Linear(in_features=128, out_features=37, bias=True)
)
Training BiLSTM Tagger...


100%|██████████| 1873/1873 [00:07<00:00, 253.23it/s]


Epoch 1: Loss = 0.2811, Valid F1 = 0.6925


100%|██████████| 1873/1873 [00:07<00:00, 258.05it/s]


Epoch 2: Loss = 0.1364, Valid F1 = 0.7227


...

Epoch 3: Loss = 0.1117, Valid F1 = 0.7252


100%|██████████| 1873/1873 [00:30<00:00, 61.94it/s]


Epoch 4: Loss = 0.0994, Valid F1 = 0.7388


100%|██████████| 1873/1873 [00:40<00:00, 46.14it/s]


Epoch 5: Loss = 0.0918, Valid F1 = 0.7337

Test F1 BiLSTM = 0.7433


### Transformer Tagger

In [29]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(0), :]
        return self.dropout(x)


class TransformerTagger(nn.Module):
    def __init__(
        self, vocab_size, label_size, embed_dim, nhead=8, num_layers=4, dropout=0.1
    ):
        super(TransformerTagger, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead, dropout=dropout
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, enable_nested_tensor=False
        )
        self.fc = nn.Linear(embed_dim, label_size)
        self.embed_dim = embed_dim
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        src_mask = src == 0  # [batch_size, src_len]
        src = self.embedding(src) * math.sqrt(self.embed_dim)
        src = self.dropout(src)
        src = src.transpose(0, 1)  # [src_len, batch_size, embed_dim]
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src, src_key_padding_mask=src_mask)
        output = output.transpose(0, 1)  # [batch_size, src_len, embed_dim]
        output = self.dropout(output)
        return self.fc(output)

# Hyperparameters
EMBED_DIM = 64
LEARNING_RATE = 1e-3
EPOCHS = 5
DROPOUT_RATE = 0.0

transformer_model = TransformerTagger(len(vocab), num_labels, EMBED_DIM, dropout=DROPOUT_RATE).to(device)
print(transformer_model)
optimizer_tf = torch.optim.Adam(transformer_model.parameters(), lr=LEARNING_RATE)

print("Training Transformer Tagger...")
for epoch in range(EPOCHS):
    loss = train_epoch(transformer_model, train_loader, optimizer_tf)
    f1 = evaluate(transformer_model, valid_loader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Valid F1 = {f1:.4f}")

test_f1_tf = evaluate(transformer_model, test_loader, export_prediction=True, model_name="transformer")
print(f"\nTest F1 Transformer = {test_f1_tf:.4f}")

TransformerTagger(
  (embedding): Embedding(41326, 64, padding_idx=0)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.0, inplace=False)
        (dropout2): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=64, out_features=37, bias=True)
  (dropout): Dropout(p=0.0, inplace=False)
)
Training Transformer Tagger

100%|██████████| 1873/1873 [00:28<00:00, 65.67it/s]


Epoch 1: Loss = 0.6178, Valid F1 = 0.3875


100%|██████████| 1873/1873 [00:28<00:00, 65.59it/s]


Epoch 2: Loss = 0.3050, Valid F1 = 0.4749


...

Epoch 3: Loss = 0.2392, Valid F1 = 0.5231


100%|██████████| 1873/1873 [00:27<00:00, 66.94it/s]


Epoch 4: Loss = 0.2041, Valid F1 = 0.5393


100%|██████████| 1873/1873 [00:28<00:00, 65.61it/s]


Epoch 5: Loss = 0.1817, Valid F1 = 0.5456

Test F1 Transformer = 0.5483


### DistilBERT Tagger

In [33]:
from transformers import DistilBertTokenizerFast, DistilBertForTokenClassification

model_name = "distilbert-base-cased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)


class BertNERDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item["tokens"]
        tags = [int(t) for t in item["tags"]]

        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
        )

        labels = []
        for i, doc_offset in enumerate(encoding["offset_mapping"]):
            if doc_offset[0] == 0 and doc_offset[1] != 0:
                labels.append(tags[encoding.word_ids()[i]])
            else:
                labels.append(-100)

        encoding.pop("offset_mapping")
        item = {key: torch.tensor(val) for key, val in encoding.items()}
        item["labels"] = torch.tensor(labels)
        return item

# Hyperparameters
BATCH_SIZE = 8
EPOCHS = 4
LEARNING_RATE = 1e-5

bert_train_dataset = BertNERDataset(train_data, tokenizer, MAX_LEN)
bert_valid_dataset = BertNERDataset(valid_data, tokenizer, MAX_LEN)
bert_test_dataset = BertNERDataset(test_data, tokenizer, MAX_LEN)

bert_train_loader = DataLoader(bert_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
bert_valid_loader = DataLoader(bert_valid_dataset, batch_size=BATCH_SIZE)
bert_test_loader = DataLoader(bert_test_dataset, batch_size=BATCH_SIZE)

bert_model = DistilBertForTokenClassification.from_pretrained(
    model_name, num_labels=num_labels
).to(device)
bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=LEARNING_RATE)


def train_bert_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        model.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_bert(model, loader, export_prediction=False, model_name="bert"):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            for i in range(labels.size(0)):
                true_tags = [id2label[t.item()] for t in labels[i] if t.item() != -100]
                pred_tags = [
                    id2label[p.item()]
                    for p, t in zip(preds[i], labels[i])
                    if t.item() != -100
                ]
                y_true.append(true_tags)
                y_pred.append(pred_tags)
    if export_prediction:
        export_predictions(model_name, y_pred)
    return seq_f1_score(y_true, y_pred)

print("Training DistilBERT Tagger...")
for epoch in range(EPOCHS):
    loss = train_bert_epoch(bert_model, bert_train_loader, bert_optimizer)
    f1 = evaluate_bert(bert_model, bert_valid_loader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Valid F1 = {f1:.4f}")

test_f1_bert = evaluate_bert(bert_model, bert_test_loader, export_prediction=True, model_name="bert")
print(f"\nTest F1 DistilBERT = {test_f1_bert:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training DistilBERT Tagger...


100%|██████████| 7491/7491 [08:16<00:00, 15.08it/s]


Epoch 1: Loss = 0.1376, Valid F1 = 0.8471


...

Epoch 2: Loss = 0.0615, Valid F1 = 0.8549


100%|██████████| 7491/7491 [08:17<00:00, 15.06it/s]


Epoch 3: Loss = 0.0434, Valid F1 = 0.8553


100%|██████████| 7491/7491 [08:15<00:00, 15.11it/s]


Epoch 4: Loss = 0.0322, Valid F1 = 0.8546

Test F1 DistilBERT = 0.8639


### Results

The best performance on the test set for each model is:

| Model | Test F1-score |
|:------|--------------:|
| BiLSTM | 0.7433 |
| Transformer | 0.5483 | 
| DistilBERT | **0.8639** |

where DistilBERT has the best performance among all models. 